### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="credit_g",
    dataset_year="1994",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5NC77",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/credit_g/ && wget -P local-data-warehouse/credit_g/ https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip && unzip local-data-warehouse/credit_g/statlog+german+credit+data.zip -d local-data-warehouse/credit_g/
""",
    # References
    academic_reference_bibtex=r"""@misc{hofmann1994statlog,
  author       = {Hofmann, H.},
  title        = {Statlog (German Credit Data) [Dataset]},
  year         = {1994},
  howpublished = {\url{https://doi.org/10.24432/C5NC77}},
  note         = {UCI Machine Learning Repository},
}
""",
    academic_reference_bibtex_key="hofmann1994statlog",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We reversed the original ordinal encoding.
- Anomaly: the original task used a cost matrix for evaluation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="good_or_bad_customer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="good_or_bad_customer",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/german.data", header=None, sep=r"\s+")

feature_names = {
    0: "checking_status",
    1: "duration_months",
    2: "credit_history",
    3: "credit_purpose",
    4: "credit_amount",
    5: "savings_status",
    6: "employment_since",
    7: "installment_rate_percent",
    8: "personal_status_sex",
    9: "other_debtors",
    10: "residence_since",
    11: "property",
    12: "age_years",
    13: "other_installment_plans",
    14: "housing",
    15: "existing_credits_count",
    16: "job",
    17: "people_liable",
    18: "telephone",
    19: "foreign_worker",
    20: "good_or_bad_customer",
}

category_mappings = {
    "checking_status": {
        "A11": "<0 DM", "A12": "0 <= ... < 200 DM",
        "A13": ">= 200 DM / salary assignments for >= 1 year",
        "A14": "no checking account",
    },
    "credit_history": {
        "A30": "no credits taken / all paid duly",
        "A31": "all credits at this bank paid duly",
        "A32": "existing credits paid duly till now",
        "A33": "delay in paying off in past",
        "A34": "critical account / other credits existing",
    },
    "credit_purpose": {
        "A40": "car (new)", "A41": "car (used)", "A42": "furniture/equipment",
        "A43": "radio/television", "A44": "domestic appliances", "A45": "repairs",
        "A46": "education", "A47": "vacation", "A48": "retraining",
        "A49": "business", "A410": "others",
    },
    "savings_status": {
        "A61": "< 100 DM", "A62": "100 <= ... < 500 DM",
        "A63": "500 <= ... < 1000 DM", "A64": ">= 1000 DM",
        "A65": "unknown / no savings",
    },
    "employment_since": {
        "A71": "unemployed", "A72": "< 1 year", "A73": "1 <= ... < 4 years",
        "A74": "4 <= ... < 7 years", "A75": ">= 7 years",
    },
    "personal_status_sex": {
        "A91": "male: divorced/separated",
        "A92": "female: divorced/separated/married",
        "A93": "male: single", "A94": "male: married/widowed",
        "A95": "female: single",
    },
    "other_debtors": {"A101": "none", "A102": "co-applicant", "A103": "guarantor"},
    "property": {
        "A121": "real estate",
        "A122": "building society savings / life insurance",
        "A123": "car or other (not savings)",
        "A124": "unknown / no property",
    },
    "other_installment_plans": {"A141": "bank", "A142": "stores", "A143": "none"},
    "housing": {"A151": "rent", "A152": "own", "A153": "for free"},
    "job": {
        "A171": "unemployed / unskilled non-resident",
        "A172": "unskilled resident",
        "A173": "skilled employee / official",
        "A174": "management / self-employed / highly qualified",
    },
    "telephone": {"A191": "none", "A192": "yes, registered"},
    "foreign_worker": {"A201": "yes", "A202": "no"},
    "good_or_bad_customer": {"1": "good", "2": "bad"},
}

df.columns = list(feature_names.values())
for feature_idx in range(len(list(df))):
    fname = feature_names[feature_idx]
    if fname in category_mappings:
        mapping = category_mappings[fname]
        df[fname] = df[fname].apply(lambda x: mapping[str(x)])

cat_features = [
    "checking_status",
    "credit_history",
    "credit_purpose",
    "savings_status",
    "employment_since",
    "personal_status_sex",
    "other_debtors",
    "property",
    "other_installment_plans",
    "housing",
    "job",
    "telephone",
    "foreign_worker",
    "good_or_bad_customer",
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,000
Columns: 21
Use sampling: False (sample size: 1,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['credit_amount', 'age_years', 'duration_months', 'credit_purpose', 'savings_status', 'credit_history', 'employment_since', 'checking_status', 'personal_status_sex', 'installment_rate_percent']
Rows remaining as candidates after top-10 filter: 0 (of 1,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,checking_status,duration_months,credit_history,credit_purpose,credit_amount,savings_status,employment_since,installment_rate_percent,personal_status_sex,other_debtors,residence_since,property,age_years,other_installment_plans,housing,existing_credits_count,job,people_liable,telephone,foreign_worker,good_or_bad_customer
0,<0 DM,18,existing credits paid duly till now,radio/television,3190,< 100 DM,1 <= ... < 4 years,2,female: divorced/separated/married,none,2,real estate,24,none,own,1,skilled employee / official,1,none,yes,bad
1,<0 DM,18,existing credits paid duly till now,car (new),4380,100 <= ... < 500 DM,1 <= ... < 4 years,3,male: single,none,4,car or other (not savings),35,none,own,1,unskilled resident,2,"yes, registered",yes,good
2,<0 DM,24,all credits at this bank paid duly,car (new),2325,100 <= ... < 500 DM,4 <= ... < 7 years,2,male: single,none,3,car or other (not savings),32,bank,own,1,skilled employee / official,1,none,yes,good
3,>= 200 DM / salary assignments for >= 1 year,12,existing credits paid duly till now,radio/television,1297,< 100 DM,1 <= ... < 4 years,3,male: married/widowed,none,4,real estate,23,none,rent,1,skilled employee / official,1,none,yes,good
4,no checking account,33,critical account / other credits existing,car (used),7253,< 100 DM,4 <= ... < 7 years,3,male: single,none,2,car or other (not savings),35,none,own,2,management / self-employed / highly qualified,1,"yes, registered",yes,good


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,checking_status,category,0.0,0.0,4.0,"no checking account, <0 DM, 0 <= ... < 200 DM, >= 200 DM / salary assignments for >= 1 year"
1,credit_history,category,0.0,0.0,5.0,"existing credits paid duly till now, critical account / other credits existing, delay in paying off in past, all credits at this bank paid duly, no credits taken / all paid duly"
2,credit_purpose,category,0.0,0.0,10.0,"radio/television, car (new), furniture/equipment, car (used), business, education, repairs, domestic appliances, others, retraining"
3,savings_status,category,0.0,0.0,5.0,"< 100 DM, unknown / no savings, 100 <= ... < 500 DM, 500 <= ... < 1000 DM, >= 1000 DM"
4,employment_since,category,0.0,0.0,5.0,"1 <= ... < 4 years, >= 7 years, 4 <= ... < 7 years, < 1 year, unemployed"
5,personal_status_sex,category,0.0,0.0,4.0,"male: single, female: divorced/separated/married, male: married/widowed, male: divorced/separated"
6,other_debtors,category,0.0,0.0,3.0,"none, guarantor, co-applicant"
7,property,category,0.0,0.0,4.0,"car or other (not savings), real estate, building society savings / life insurance, unknown / no property"
8,other_installment_plans,category,0.0,0.0,3.0,"none, bank, stores"
9,housing,category,0.0,0.0,3.0,"own, rent, for free"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
duration_months,1000.0,20.903,12.058814,4.0,72.0
credit_amount,1000.0,3271.258,2822.736876,250.0,18424.0
installment_rate_percent,1000.0,2.973,1.118715,1.0,4.0
residence_since,1000.0,2.845,1.103718,1.0,4.0
age_years,1000.0,35.546,11.375469,19.0,75.0
existing_credits_count,1000.0,1.407,0.577654,1.0,4.0
people_liable,1000.0,1.155,0.362086,1.0,2.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                  rank                                                  
checking_status         1                               no checking account   
                        2                                             <0 DM   
                        3                                 0 <= ... < 200 DM   
                        4      >= 200 DM / salary assignments for >= 1 year   
credit_history          1               existing credits paid duly till now   
                        2         critical account / other credits existing   
                        3                       delay in paying off in past   
                        4                all credits at this bank paid duly   
                        5                  no credits taken / all paid duly   
credit_purpose          1                                  radio/television   
                        2                                         car (new)   
                        3                               furniture/equipment   
                        4                                        car (used)   
                        5                                          business   
employment_since        1                                1 <= ... < 4 years   
                        2                                        >= 7 years   
                        3                                4 <= ... < 7 years   
                        4                                          < 1 year   
                        5                                        unemployed   
foreign_worker          1                                               yes   
                        2                                                no   
good_or_bad_customer    1                                              good   
                        2                                               bad   
housing                 1                                               own   
                        2                                              rent   
                        3                                          for free   
job                     1                       skilled employee / official   
                        2                                unskilled resident   
                        3     management / self-employed / highly qualified   
                        4               unemployed / unskilled non-resident   
other_debtors           1                                              none   
                        2                                         guarantor   
                        3                                      co-applicant   
other_installment_plans 1                                              none   
                        2                                              bank   
                        3                                            stores   
personal_status_sex     1                                      male: single   
                        2                female: divorced/separated/married   
                        3                             male: married/widowed   
                        4                          male: divorced/separated   
property                1                        car or other (not savings)   
                        2                                       real estate   
                        3         building society savings / life insurance   
                        4                             unknown / no property   
savings_status          1                                          < 100 DM   
                        2                              unknown / no savings   
                        3                               100 <= ... < 500 DM   
                        4                              500 <= ... < 1000 DM   
                        5                                        >= 1000 DM   
telephone               1                

In [8]:
# Target Distribution
target_df

,count,pct
good_or_bad_customer,,
good,700,70.0
bad,300,30.0


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to credit_g/019d5a3e-45e9-73c3-8bbc-51310c58e706
019d5a3e-45e9-73c3-8bbc-51310c58e706
281eaf98fd9c14f9999bb33cf61002d553ec1d64a53c86d4435030602d466f49
